In [ ]:
!pip install torch torchaudio transformers librosa soundfile numpy

In [ ]:
import os
import math
import numpy as np
import librosa
import soundfile as sf
import torch
import torchaudio
from transformers import pipeline


class AudioProcessor:
    def __init__(self, target_sr: int = 16000, chunk_duration: int = 10):
        self.target_sr = target_sr
        self.chunk_duration = chunk_duration

    def load_audio(self, file_path: str):
        """
        Load audio and resample to target sample rate.
        """
        audio, sr = librosa.load(file_path, sr=None, mono=True)

        if sr != self.target_sr:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=self.target_sr)
            sr = self.target_sr

        return audio, sr

    def normalize_audio(self, audio: np.ndarray):
        """
        Normalize audio amplitude for consistency.
        """
        max_val = np.max(np.abs(audio))
        if max_val > 0:
            audio = audio / max_val
        return audio

    def extract_mfcc(self, audio: np.ndarray, sr: int, n_mfcc: int = 13):
        """
        Convert audio into numerical features (MFCCs).
        """
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
        return mfccs

    def chunk_audio(self, audio: np.ndarray, sr: int):
        """
        Split audio into manageable chunks.
        """
        chunk_size = sr * self.chunk_duration
        chunks = []

        for start in range(0, len(audio), chunk_size):
            end = start + chunk_size
            chunk = audio[start:end]
            if len(chunk) > 0:
                chunks.append(chunk)

        return chunks


class SpeechToTextModel:
    def __init__(self, model_name: str = "openai/whisper-base"):
        """
        Uses Hugging Face ASR pipeline.
        """
        device = 0 if torch.cuda.is_available() else -1
        self.asr = pipeline(
            "automatic-speech-recognition",
            model=model_name,
            device=device
        )

    def transcribe_chunk(self, chunk: np.ndarray, sr: int):
        """
        Transcribe a single audio chunk.
        """
        result = self.asr({"array": chunk, "sampling_rate": sr})
        return result["text"].strip()

    def transcribe_audio(self, chunks, sr: int):
        """
        Transcribe all chunks and combine results.
        """
        transcripts = []
        for i, chunk in enumerate(chunks):
            text = self.transcribe_chunk(chunk, sr)
            transcripts.append(text)
        return " ".join(transcripts).strip()


class WERCalculator:
    @staticmethod
    def _edit_distance(ref_words, hyp_words):
        """
        Compute edit distance matrix for WER.
        """
        rows = len(ref_words) + 1
        cols = len(hyp_words) + 1
        dp = [[0] * cols for _ in range(rows)]

        for i in range(rows):
            dp[i][0] = i
        for j in range(cols):
            dp[0][j] = j

        for i in range(1, rows):
            for j in range(1, cols):
                if ref_words[i - 1] == hyp_words[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1]
                else:
                    substitution = dp[i - 1][j - 1] + 1
                    insertion = dp[i][j - 1] + 1
                    deletion = dp[i - 1][j] + 1
                    dp[i][j] = min(substitution, insertion, deletion)

        return dp

    @staticmethod
    def compute_wer(reference: str, hypothesis: str):
        """
        Compute Word Error Rate.
        WER = (S + D + I) / N
        """
        ref_words = reference.lower().split()
        hyp_words = hypothesis.lower().split()

        if len(ref_words) == 0:
            return 0.0 if len(hyp_words) == 0 else 1.0

        dp = WERCalculator._edit_distance(ref_words, hyp_words)

        i = len(ref_words)
        j = len(hyp_words)

        substitutions = 0
        insertions = 0
        deletions = 0

        while i > 0 or j > 0:
            if i > 0 and j > 0 and ref_words[i - 1] == hyp_words[j - 1]:
                i -= 1
                j -= 1
            else:
                current = dp[i][j]

                if i > 0 and j > 0 and dp[i - 1][j - 1] + 1 == current:
                    substitutions += 1
                    i -= 1
                    j -= 1
                elif j > 0 and dp[i][j - 1] + 1 == current:
                    insertions += 1
                    j -= 1
                else:
                    deletions += 1
                    i -= 1

        wer = (substitutions + insertions + deletions) / len(ref_words)

        return {
            "wer": wer,
            "substitutions": substitutions,
            "insertions": insertions,
            "deletions": deletions,
            "reference_length": len(ref_words)
        }


def save_temp_chunk(chunk: np.ndarray, sr: int, temp_path: str = "temp_chunk.wav"):
    """
    Optional helper if you want to save chunks for debugging.
    """
    sf.write(temp_path, chunk, sr)


def process_single_file(audio_path: str, transcript_path: str, model_name: str = "openai/whisper-base"):
    processor = AudioProcessor(target_sr=16000, chunk_duration=10)
    model = SpeechToTextModel(model_name=model_name)

    audio, sr = processor.load_audio(audio_path)
    audio = processor.normalize_audio(audio)

    mfccs = processor.extract_mfcc(audio, sr)
    chunks = processor.chunk_audio(audio, sr)

    predicted_text = model.transcribe_audio(chunks, sr)

    with open(transcript_path, "r", encoding="utf-8") as f:
        actual_text = f.read().strip()

    wer_result = WERCalculator.compute_wer(actual_text, predicted_text)

    return {
        "file": os.path.basename(audio_path),
        "mfcc_shape": mfccs.shape,
        "predicted_text": predicted_text,
        "actual_text": actual_text,
        "wer_result": wer_result
    }


def evaluate_dataset(audio_folder: str, transcript_folder: str, model_name: str = "openai/whisper-base"):
    """
    Evaluate all audio files in a folder.
    Assumes:
      audio_folder/file1.wav
      transcript_folder/file1.txt
    """
    results = []
    total_wer = 0
    count = 0

    supported_extensions = (".wav", ".mp3", ".flac", ".m4a", ".ogg")

    for file_name in os.listdir(audio_folder):
        if not file_name.lower().endswith(supported_extensions):
            continue

        audio_path = os.path.join(audio_folder, file_name)
        transcript_name = os.path.splitext(file_name)[0] + ".txt"
        transcript_path = os.path.join(transcript_folder, transcript_name)

        if not os.path.exists(transcript_path):
            print(f"Skipping {file_name}: no matching transcript found.")
            continue

        try:
            result = process_single_file(audio_path, transcript_path, model_name=model_name)
            results.append(result)
            total_wer += result["wer_result"]["wer"]
            count += 1

            print("=" * 60)
            print(f"File: {result['file']}")
            print(f"MFCC shape: {result['mfcc_shape']}")
            print(f"Predicted: {result['predicted_text']}")
            print(f"Actual:    {result['actual_text']}")
            print(f"WER:       {result['wer_result']['wer']:.4f}")
            print(f"S: {result['wer_result']['substitutions']}, "
                  f"I: {result['wer_result']['insertions']}, "
                  f"D: {result['wer_result']['deletions']}")
        except Exception as e:
            print(f"Error processing {file_name}: {e}")

    average_wer = total_wer / count if count > 0 else None

    print("\n" + "=" * 60)
    print("FINAL DATASET RESULTS")
    if average_wer is not None:
        print(f"Files evaluated: {count}")
        print(f"Average WER: {average_wer:.4f}")
    else:
        print("No valid files were evaluated.")

    return results, average_wer


if __name__ == "__main__":
    # Example folder structure:
    # project/
    #   voice_to_text.py
    #   data/
    #       audio/
    #           sample1.wav
    #           sample2.wav
    #       transcripts/
    #           sample1.txt
    #           sample2.txt

    audio_folder = "data/audio"
    transcript_folder = "data/transcripts"

    evaluate_dataset(
        audio_folder=audio_folder,
        transcript_folder=transcript_folder,
        model_name="openai/whisper-base"
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

/tmp/ipykernel_3663/1313794602.py:20: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, sr=None, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Error processing sample1.wav: 'num_frames'

FINAL DATASET RESULTS
No valid files were evaluated.


In [ ]:
import os

os.makedirs("data/audio", exist_ok=True)
os.makedirs("data/transcripts", exist_ok=True)

In [ ]:
import shutil

shutil.move("sample1.wav", "data/audio/sample1.wav")
shutil.move("sample1.txt", "data/transcripts/sample1.txt")

'data/transcripts/sample1.txt'

In [ ]:
evaluate_dataset(
    audio_folder="data/audio",
    transcript_folder="data/transcripts",
    model_name="openai/whisper-base"
)

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Error processing sample1.wav: 'num_frames'

FINAL DATASET RESULTS
No valid files were evaluated.


/tmp/ipykernel_3663/1313794602.py:20: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, sr=None, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


([], None)